In [ ]:
import chess.pgn
import chess.engine
from pathlib import Path
import shutil
from tqdm.notebook import tqdm  # progress bar in Jupyter

# Auto-detect Homebrew stockfish; fallback to common paths.
ENGINE_PATH = (
    shutil.which("stockfish")
    or "/opt/homebrew/bin/stockfish"      # Apple Silicon default
    or "/usr/local/bin/stockfish"         # Intel default (will be used if previous is absent)
)

print("Using engine:", ENGINE_PATH)

In [ ]:
from lichess_analyser.game_loader import LichessGameLoader

## Select a subset of games

In [ ]:
# Only select the last 10 games
pgn_file = Path("data/lichess_games.pgn")
with pgn_file.open() as f:
    games = []
    while True:
        game = chess.pgn.read_game(f)
        if game is None:
            break
        games.append(game)
    games = games[-10:]  # Keep only the last 10 games

games

In [ ]:
# For each of the games, determine if I blundered in the opening, middlegame, or endgame
engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
blunders = []
for game in tqdm(games, desc="Analyzing games"):
    board = game.board()
    for move in game.mainline_moves():
        board.push(move)
        info = engine.analyse(board, chess.engine.Limit(time=0.1))
        score = info["score"].white().score(mate_score=10000)
        # Here you would implement your blunder detection logic
        # For simplicity, let's say a blunder is a drop of more than 200 centipawns
        # compared to the previous evaluation
        # (This is a placeholder; real blunder detection would be more complex)
        # Store blunder information if detected
    # Append results to blunders list

In [ ]:

OPENING_MOVES = 12  # opening stage length (plies)

engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)

with open(pgn_file) as pgn_handle:
    for pgn in tqdm(iter(lambda: chess.pgn.read_game(pgn_handle), None), desc="Games"):
        if pgn is None:
            break
        board = pgn.board()
        last_eval = 0
        for i, move in enumerate(pgn.mainline_moves()):
            board.push(move)
            info = engine.analyse(board, chess.engine.Limit(depth=15))
            last_eval = info["score"].white().score(mate_score=10000)
            if i + 1 >= OPENING_MOVES:  # reached end of opening stage
                break

        # Blunder definition: at end of opening stage be down ≥150 centipawns (or mating disadvantage)
        if last_eval is not None and last_eval <= -150:
            print(pgn.headers.get("Event"), pgn.headers.get("Opening"), last_eval, pgn.headers.get("Site"))

engine.quit()

In [ ]:

PLAYER_NAME = "Thijs1337"

def result_from_perspective(headers, player):
    """
    Returns numeric score from player's perspective:
    1.0 win, 0.5 draw, 0.0 loss.
    """
    res = headers.get("Result")
    white = headers.get("White")
    black = headers.get("Black")
    if res == "1-0":
        return 1.0 if white == player else 0.0
    if res == "0-1":
        return 1.0 if black == player else 0.0
    if res in ("1/2-1/2", "½-½"):
        return 0.5
    return None  # unknown/unfinished

def is_loss(headers, player):
    score = result_from_perspective(headers, player)
    return score == 0.0

# List losses
losses = []
with open(Path("data/lichess_games.pgn")) as f:
    while True:
        game = chess.pgn.read_game(f)
        if game is None:
            break
        if is_loss(game.headers, PLAYER_NAME):
            losses.append({
                "Date": game.headers.get("Date"),
                "Site": game.headers.get("Site"),
                "Event": game.headers.get("Event"),
                "Result": game.headers.get("Result"),
                "White": game.headers.get("White"),
                "Black": game.headers.get("Black"),
                "Opening": game.headers.get("Opening"),
            })

print(f"Total losses for {PLAYER_NAME}: {len(losses)}")
for g in losses[:10]:  # show first 10
    print(g)

# Load games

# Extract information from games
## Extract game phases
## Extract when the player lost
## Extract when player blundered